# 17.3 每個權重只差0.2，輸出也只差0.2嗎？


同一權重表的兩列是`[0.7,0.2]`與`[-0.7,1.2]`。用刻度0.5還原成`[0.5,0]`與`[-0.5,1]`，每格絕對差都是0.2。現在拿兩筆輸入`[1,1]`與`[1,−1]`作比較。

第一筆第一列的兩項誤差會相加，輸出差0.4；第二列的正負誤差抵消，差0。換輸入方向，則換另一列相加。線性層是權重乘輸入再相加，因此單格權重誤差不能直接當輸出上限。


In [1]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [2]:
import torch

w = torch.tensor([[0.7, 0.2], [-0.7, 1.2]])
scale = 0.5
restored = (w / scale).round() * scale
x = torch.tensor([[1.0, 1.0], [1.0, -1.0]])
original_output = x @ w.T
compressed_output = x @ restored.T
print("權重平均絕對差", round((w - restored).abs().mean().item(), 4))
print("原輸出", original_output.round(decimals=4).tolist())
print("還原權重輸出", compressed_output.round(decimals=4).tolist())
print("逐項輸出差", (original_output - compressed_output).round(decimals=4).tolist())

權重平均絕對差 0.2
原輸出 [[0.8999999761581421, 0.5], [0.5, -1.899999976158142]]
還原權重輸出 [[0.5, 0.5], [0.5, -1.5]]
逐項輸出差 [[0.4000000059604645, 0.0], [0.0, -0.4000000059604645]]


`x @ w.T`讓每筆輸入與每列權重做點積，得到兩筆、各兩個輸出。`abs().mean()`先取絕對值再平均，叫**平均絕對誤差（MAE）**，避免正負抵消。權重MAE為0.2，但逐項輸出差約`[[0.4,0],[0,−0.4]]`。

這次restored仍是浮點tensor，沒有存成int8，目的是隔離數值誤差。兩種輸出也可能打印小數長尾，先核對正負與約0、0.4的差別即可。

公式是`輸出差 = x @ (w−restored).T`。輸入的大小與方向都會影響誤差；經過更多層和非線性後，還要看最終答案，不能只憑權重MAE保證品質。

練習只在運算前將x乘10。權重MAE仍0.2，非零輸出差變成±4。相同壓縮權重在不同輸入尺度下，可以有不同影響。
